# classification ANN 

In [1]:
import pandas as pd 
import numpy as np


In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
x = df.drop("Class",axis=1)
y = df["Class"]

In [6]:
df["Class"].unique()

<ArrowStringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)


In [8]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state = 42 
)


In [9]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [11]:
x_train_tensor = torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)

x_test_tensor =  torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype= torch.long)




In [12]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)


In [13]:
train_loader = DataLoader(train_dataset, batch_size=32,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32 )

In [14]:
# built our model
class Ann(nn.Module):
    def __init__(self): 
        super(Ann,self).__init__()
        self.model = nn.Sequential(
            nn.Linear(x.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )   

    def forward(self,x):
        return self.model(x)
        
          

In [15]:
model = Ann()
#loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [16]:
#trainin the nural networks
epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss =0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        
        outputs=model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() # weight updation step

        running_loss += loss.item()

    train_loss = running_loss/len(train_loader)

    print(f"epoch={epoch+1}/{epochs}, loss = {train_loss}")

epoch=1/100, loss = 1.719877522924672
epoch=2/100, loss = 1.1037115532418955
epoch=3/100, loss = 0.7096419373284215
epoch=4/100, loss = 0.5305819355923197
epoch=5/100, loss = 0.4339860042800074
epoch=6/100, loss = 0.36766042851883435
epoch=7/100, loss = 0.33186676320822345
epoch=8/100, loss = 0.29599140001379926
epoch=9/100, loss = 0.2698525831751201
epoch=10/100, loss = 0.2583117805745291
epoch=11/100, loss = 0.2419641351570254
epoch=12/100, loss = 0.22652974886738736
epoch=13/100, loss = 0.20305192373369052
epoch=14/100, loss = 0.1954294958192369
epoch=15/100, loss = 0.1974084701227105
epoch=16/100, loss = 0.17764453492734744
epoch=17/100, loss = 0.17326055126993553
epoch=18/100, loss = 0.16599548081664936
epoch=19/100, loss = 0.16133439233121666
epoch=20/100, loss = 0.15078114118912947
epoch=21/100, loss = 0.14866515751118245
epoch=22/100, loss = 0.15132044441998005
epoch=23/100, loss = 0.1364391928781634
epoch=24/100, loss = 0.13209199030762134
epoch=25/100, loss = 0.13238330266397

In [20]:
# evaluate 
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted=torch.max(outputs,1)   # returns max value of the tensor and the inidex of the max valued tensor 

        correct += (predicted == yb).sum().item()
        total += yb.size(0)    # actual samples in  the batch 
        
print("total val:", total)
print("correct val:",correct )

accuracy = correct/total  *100

print("accuracy of the ann :",accuracy)

total val: 180
correct val: 168
accuracy of the ann : 93.33333333333333


In [ ]:
# do pca on this 